In [ ]:
import pandas as pd
import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

## Adding qid

In [ ]:
query_df = pd.read_csv("Data/query_formulation_df.csv")
serp_df = pd.read_csv("Data/logs_compiled.csv")
queries = pd.read_csv("Data/query_df.csv")
query_df["qid"] = [serp_df.loc[serp_df["serp_id"]==serp_id]["qid"].values[0] for serp_id in query_df["serp_id"]]
query_df["timestamp"] = [queries.loc[queries["qid"]==qid]["timestamp"].values[0] for qid in query_df["qid"]]
query_df = query_df[["uid","topic_number","qid","query", "timestamp", "serp_id", "precision@10"]]
query_df = query_df.sort_values(["uid", "timestamp"]).reset_index(drop=True)
query_df.to_csv("Data/query_formulation_df.csv", index=False, encoding="utf8")
query_df.head(10)

In [ ]:
query_df.loc[query_df["qid"]=="q10"]

## Analysing queries

In [ ]:
import pandas as pd
import numpy as np
from driver import get_text_stats
import language_tool_python
from Levenshtein import distance
from tqdm.notebook import tqdm
import spacy

nlp = spacy.load("it_core_news_lg")
tool = language_tool_python.LanguageTool('it')

query_df = pd.read_csv("Data/query_formulation_df.csv")
concreteness = pd.read_excel("Resources/IT_concreteness_database.xlsx")
text_stat_results = get_text_stats(query_df, "query")

query_df["query_len"] = text_stat_results["content_len"]
query_df["mean_aoa"] = text_stat_results["mean_aoa"]
query_df["mean_conc"] = text_stat_results["mean_conc"]
query_df["gulpease_index"] = text_stat_results["gulpease_index"]

corrected_query_list = []
num_typos = []
levenshtein_dist = []
lex_density = []
lex_diversity = []
ttr = []
stop_word_ratio = []
num_concepts = []
concepts = []
mean_concepts_conc = []
# query_len = []


for _, row in tqdm(query_df.iterrows(), total=len(query_df)):
    query = row["query"]
    corrected_query = tool.correct(query)
    if query.lower() == corrected_query.lower():
        corrected_query_list.append("")
        num_typos.append(0)
    else:
        corrected_query_list.append(corrected_query)
        matches = tool.check(query)
        num_typos.append(len(matches))
    levenshtein_dist.append(distance(query.lower(), corrected_query.lower()))
    doc = nlp(query)

    lex_words = [] # lexical words, i.e., nouns, verbs, adjectives, and adverbs

    words = []

    # Extract noun chunks
    noun_phrases = [chunk.text for chunk in doc.noun_chunks]

    # get the concreteness of each noun chunk
    noun_phrases_conc = []
    for phrase in noun_phrases:
        phrase_conc = []
        phrase_doc = nlp(phrase)
        for phrase_token in phrase_doc:
            if phrase_token.text.lower() in concreteness["Ita_Word"].tolist():
                phrase_conc.append(concreteness.loc[concreteness["Ita_Word"]==phrase_token.text.lower()]["M_Con"].values[0])
        if len(phrase_conc) == 0: # none of the words in the noun chunk have a concreteness value
            noun_phrases_conc.append(None)
        else:
            noun_phrases_conc.append(np.mean(phrase_conc))

    stop_word_count = 0
    for token in doc:
        # print(token.text, token.pos_, token.tag_)
        if token.pos_ in ["ADJ", "ADV", "NOUN", "PROPN", "VERB", "AUX"]:
            lex_words.append(token.text.strip().lower())
        if token.pos_ not in ["SYM", "X", "PUNCT"]:
            words.append(token.text.strip().lower())
        if token.is_stop:
            stop_word_count += 1

    assert len(words) == row["query_len"] # number of words should be same as query length

    # query_len.append(len(words)-stop_word_count) # query length = number of non stop words
    lex_density.append(len(lex_words)/len(words))
    ttr.append(len(set(words))/len(words)) # type token ratio: num of unique words/total num of words
    lex_diversity.append(len(set(lex_words))/len(lex_words)) # lexical diversity: num of unique lexical words/total num of lexical words
    stop_word_ratio.append(stop_word_count/len(words))
    num_concepts.append(len(noun_phrases))
    concepts.append(noun_phrases)
    mean_concepts_conc.append(pd.Series(noun_phrases_conc).dropna().mean())
    # print(query, corrected_query, len(matches), distance(query, corrected_query))

# query_df["query_len"] = query_len
query_df["lex_density"] = lex_density
query_df["lex_diversity"] = lex_diversity
query_df["ttr"] = ttr
query_df["stop_word_ratio"] = stop_word_ratio
query_df["corrected_query"] = corrected_query_list
query_df["num_typos"] = num_typos
query_df["levenshtein_dist"] = levenshtein_dist
query_df["num_concepts"] = num_concepts
query_df["concepts"] = concepts
query_df["mean_concepts_conc"] = mean_concepts_conc

query_df["effective_query"] = [int(p10 >= 0.8) for p10 in query_df["precision@10"]]

In [ ]:
query_df.head(2)

In [ ]:
query_df.to_csv("Data/query_formulation_df.csv", index=False, encoding="utf8")
query_df.head()

In [ ]:
query_df_sample = query_df.sample(5, random_state=67).sort_index()
query_df_sample

In [ ]:
query_df.loc[query_df["effective_query"]==1]

## Visualisation

In [ ]:
import pandas as pd

query_df = pd.read_csv("Data/query_formulation_df.csv")
categories = [
    # language fluency
    "lex_density", "mean_aoa", "gulpease_index", 
    # "lex_diversity", "ttr", 
    "num_typos", "levenshtein_dist",

    # query intent complexity
    "num_concepts", "mean_concepts_conc", "query_len", "stop_word_ratio", 
    # "mean_conc",
    ]

In [ ]:
len(query_df.loc[query_df["effective_query"]==1])

### RQ1

In [ ]:
from driver import get_descriptive_stats

binwidth_map = {
    "lex_density": 0.1,
    # "lex_diversity": 0.01,
    # "ttr": 0.1,
    "mean_aoa": 0.5,
    "gulpease_index": 20,
    "num_typos": 1,
    "levenshtein_dist": 1,
    "query_len": 1,
    "stop_word_ratio": 0.1,
    "num_concepts": 1,
    # "mean_conc": 0.5,
    "mean_concepts_conc": 0.5
}

desc_stats = get_descriptive_stats(df=query_df, search_stage="query_formulation", categories=categories, ncols=3, nrows=3, binwidth_map=binwidth_map)
desc_stats

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(15,3))
plt.subplots_adjust(hspace=0.85)
for i, topic_num in enumerate(["g3t1", "g3t2", "g3t3", "g4t1", "g4t2", "g4t3"]):
    ax = plt.subplot(2,3,i+1)
    topic_query_df = query_df.loc[query_df["topic_number"] == topic_num]
    med = topic_query_df["query_len"].median()
    sns.histplot(data=topic_query_df, x="query_len", binwidth=1, discrete=True, ax=ax, stat="proportion", kde=True, alpha=0.2)
    ax.axvline(med, linestyle="--", color="red")
    ax.set_title(topic_num)
    ax.set_xlabel("")

plt.show()

In [ ]:
query_df["query_len"].value_counts()

In [ ]:
len(query_df)

In [ ]:
val0 = len(query_df.loc[query_df["gulpease_index"]<80]["gulpease_index"].dropna())/len(query_df["gulpease_index"].dropna())
print(f"Ratio of queries not readable for primary school students: {val0:.2f}")

val1 = len(query_df.loc[query_df["gulpease_index"]<40]["gulpease_index"].dropna())/len(query_df["gulpease_index"].dropna())
print(f"Ratio of queries not readable for even high school graduates: {val1:.2f}")

val2 = len(query_df.loc[(query_df["gulpease_index"]<60)&(query_df["gulpease_index"]>=40)]["gulpease_index"].dropna())/len(query_df["gulpease_index"].dropna())
print(f"Ratio of queries not readable for middle school graduates but readable for high school students: {val2:.2f}")

val3 = len(query_df.loc[(query_df["gulpease_index"]<80)&(query_df["gulpease_index"]>=60)]["gulpease_index"].dropna())/len(query_df["gulpease_index"].dropna())
print(f"Ratio of queries readable for middle school students but not primary school students: {val3:.2f}")

val4 = len(query_df.loc[(query_df["gulpease_index"]>=80)]["gulpease_index"].dropna())/len(query_df["gulpease_index"].dropna())
print(f"Ratio of queries readable for primary school students: {val4:.2f}")

In [ ]:
query_df["mean_concepts_conc"].value_counts()/len(query_df["mean_concepts_conc"].dropna())

In [ ]:
query_df["query_len"].value_counts()/len(query_df["query_len"].dropna())

In [ ]:
query_df.loc[query_df["num_typos"]>0]["num_typos"].value_counts()

In [ ]:
mistakes = len(query_df.loc[query_df["num_typos"]>0]["num_typos"].dropna())/len(query_df["num_typos"].dropna())

print(f"Ratio of queries with spelling errors: {mistakes:.2f}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(16,9))
plt.subplots_adjust(hspace=0.45)
for ax, category in zip(axes.flatten(), categories):
    # sns.kdeplot(data=query_df, x=category, ax=ax, cut=0, fill=True, alpha=0.25)
    sns.histplot(data=query_df, x=category, ax=ax, kde=True, alpha=0.1, stat="proportion", kde_kws={"cut":0})
    # mean = query_df[category].dropna().mean()
    # std_dev = query_df[category].dropna().std()
    # ax.axvline(mean, color="red", linestyle=":", label="mean")
    # ax.axvspan(max(mean-std_dev, query_df[category].min()), min(mean+std_dev, query_df[category].max()), color="red", alpha=0.25, label="mean ± std")

    # median = query_df[category].dropna().median()
    # mad = (query_df[category].dropna() - median).abs().median()
    # ax.axvline(median, color="green", linestyle="--", label="median")
    # ax.axvspan(max(median-mad, query_df[category].min()), min(median+mad, query_df[category].max()), color="green", alpha=0.25, label="median ± MAD")
    
    ax.set_xlabel(category)
handles, labels = ax.get_legend_handles_labels()
plt.figlegend(handles, labels, loc="upper center", ncol=4, fontsize=20)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import gaussian_kde
from scipy.signal import find_peaks


fig, axes = plt.subplots(ncols=3, nrows=3, figsize=(16,9))

for ax, col in zip(axes.flatten(), categories):

    x = query_df[col].dropna().values

    # KDE
    kde = gaussian_kde(x)
    # xs = np.linspace(x.min(), x.max(), 1000)
    xs = np.sort(x)
    ys = kde(xs)

    # Find local maxima
    peak_idx, props = find_peaks(
        ys,
        prominence=0.05 * ys.max(), # adjust as needed
        # distance=int(0.1*len(xs))
    )

    peaks = xs[peak_idx]

    if len(peaks) == 0:
        peaks = np.array([xs[np.argmax(ys)]])

    # Assign each observation to nearest peak
    peak_assignments = peaks[np.argmin(np.abs(x[:, None] - peaks), axis=1)]

    # Plot KDE
    # fig, ax = plt.subplots(figsize=(10, 5))

    ax.plot(xs, ys, color="tab:blue", lw=2)
    ax.fill_between(xs, ys, alpha=0.2, color="tab:blue")

    for peak in peaks:

        cluster = x[peak_assignments == peak]

        # # Skip tiny clusters
        # if len(cluster) < 10:
        #     continue

        cluster_std = np.std(cluster)

        # p10 = np.percentile(cluster, 10)
        # p90 = np.percentile(cluster, 90)

        # Shade local 80% interval
        ax.axvspan(
            peak-cluster_std,
            peak+cluster_std,
            alpha=0.15
        )

        # Mark mode
        peak_height = kde(peak)[0]

        ax.plot(
            peak,
            peak_height,
            "ko",
            markersize=6
        )

        ax.axvline(
            peak,
            color="black",
            linestyle="--",
            alpha=0.7
        )

        ax.text(
            peak,
            peak_height * 1.05,
            f"{peak:.2f}",
            ha="center"
        )

    ax.set_title(f"{col}: Modes and Local 80% Intervals")
    ax.set_ylabel("Density")

plt.tight_layout()
plt.show()

### RQ2

In [ ]:
from driver import get_significance

desc_stats_df, sig_test_results = get_significance(df=query_df, effective_label="effective_query", search_stage="query_formulation", categories=categories)
sig_test_results

In [ ]:
desc_stats_df

In [ ]:
from driver import get_boxplots

get_boxplots(query_df, "query_formulation", categories, "effective_query", artefact="query", n_cols=3, n_rows=3, stats_df=sig_test_results)

In [ ]:
# from scipy.stats import spearmanr

# for category in categories:
#     res = spearmanr(query_df["precision@10"], query_df[category], nan_policy="omit")
#     print(f"Category: {category}; \tcoefficient: {res.statistic}; \tpval: {res.pvalue}")